# Two no-MMD-loss training runs: summary comparison

Both columns use networks trained without MMD loss. The left column is run 1; the right is run 2 (`noMMD_rerun1`).

The source figures are regenerated by `summary_noMMDloss_runs_M0_M3_4x4.ipynb` with diagnostic score $\rho=d_M/d_{M,\mathrm{high}}$. This notebook combines their saved PNGs and validates the corresponding calibration tables.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display
from PIL import Image as PILImage, ImageDraw, ImageFont
from matplotlib import font_manager

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents) if (path / "benchmark").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from benchmark.examples.diffusion.config import BASE_DIR
from benchmark.examples.diffusion.results.summary_dimension_comparison import (
    VISUALIZATION_SUMMARY_LABELS,
)

FIGURE_DIR = PROJECT_ROOT / "benchmark/examples/diffusion/results/plots"
ORIGINAL_DIR = FIGURE_DIR / "summary_noMMDloss_runs_M0_M3_4x4/run1"
RERUN_DIR = FIGURE_DIR / "summary_noMMDloss_runs_M0_M3_4x4/run2"
OUTPUT_DIR = FIGURE_DIR / "summary_original_vs_rerun1"
RUN1_THRESHOLDS = BASE_DIR / "calibration_outputs_100_noMMD" / "thresholds.csv"
RUN2_THRESHOLDS = BASE_DIR / "calibration_outputs_100_noMMD_rerun1" / "thresholds.csv"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PLOTS = {
    "pmp_error_gold_standard_overview_8x4": "Gold-standard PMP comparison",
    "log10_logml_error_M0_M3_by_diagnostic": "Log marginal-likelihood error comparison",
    "pmp_error_M0_M3_by_diagnostic": "Signed PMP error comparison",
    "posterior_mmd_M0_M3_by_diagnostic": "Raw posterior MMD comparison",
}

FONT_PATH = font_manager.findfont("DejaVu Sans")
HEADER_FONT = ImageFont.truetype(FONT_PATH, 58)
ROW_FONT = ImageFont.truetype(FONT_PATH, 48)

THRESHOLD_CONTRACT = {
    "posterior_mmd": (0.95, 0.00, 0.95),
    "signed_logml_error": (0.90, 0.05, 0.95),
    "signed_pmp_error": (0.90, 0.05, 0.95),
}


def validate_threshold_contract(path):
    table = pd.read_csv(path)
    for metric, expected in THRESHOLD_CONTRACT.items():
        actual = table.loc[
            table["metric"].eq(metric),
            ["quantile", "lower_quantile", "upper_quantile"],
        ].drop_duplicates()
        if len(actual) != 1 or not np.allclose(actual.iloc[0], expected):
            raise ValueError(f"{path}: unexpected interval for {metric}:\n{actual}")
    if tuple(VISUALIZATION_SUMMARY_LABELS) != ("S=D", "S=2D", "S=4D"):
        raise ValueError("Summary plots must show D/2D/4D only.")
    return table


RUN1_THRESHOLD_TABLE = validate_threshold_contract(RUN1_THRESHOLDS)
RUN2_THRESHOLD_TABLE = validate_threshold_contract(RUN2_THRESHOLDS)
print("Validated both run-specific threshold files and the D/2D/4D plot filter.")

In [ ]:
def _centered_text(draw, box, text, font):
    left, top, right, bottom = box
    text_box = draw.textbbox((0, 0), text, font=font)
    width = text_box[2] - text_box[0]
    height = text_box[3] - text_box[1]
    draw.text(
        ((left + right - width) / 2, (top + bottom - height) / 2),
        text,
        fill="black",
        font=font,
    )


def stitch_pair(name, title):
    left_path = ORIGINAL_DIR / f"{name}.png"
    right_path = RERUN_DIR / f"{name}.png"
    missing = [str(path) for path in (left_path, right_path) if not path.exists()]
    if missing:
        raise FileNotFoundError(
            "Missing saved source figures for this comparison:\n" + "\n".join(missing)
        )
    left = PILImage.open(left_path).convert("RGB")
    right = PILImage.open(right_path).convert("RGB")
    if right.size != left.size:
        right = right.resize(left.size, PILImage.Resampling.LANCZOS)
    gap = 36
    header_height = 130
    canvas = PILImage.new(
        "RGB",
        (left.width + gap + right.width, header_height + left.height),
        "white",
    )
    draw = ImageDraw.Draw(canvas)
    _centered_text(
        draw,
        (0, 0, left.width, header_height),
        "No-MMD training: run 1",
        HEADER_FONT,
    )
    _centered_text(
        draw,
        (left.width + gap, 0, canvas.width, header_height),
        "No-MMD training: run 2 (rerun1)",
        HEADER_FONT,
    )
    canvas.paste(left, (0, header_height))
    canvas.paste(right, (left.width + gap, header_height))
    output = OUTPUT_DIR / f"{name}_side_by_side.png"
    canvas.save(output, dpi=(220, 220))
    return {"title": title, "path": output, "image": canvas}


paired = {name: stitch_pair(name, title) for name, title in PLOTS.items()}

gap = 36
top_header = 130
row_header = 100
pair_width = max(item["image"].width for item in paired.values())
total_height = top_header + sum(
    row_header + item["image"].height - top_header for item in paired.values()
)
combined = PILImage.new("RGB", (pair_width, total_height), "white")
draw = ImageDraw.Draw(combined)
half_width = (pair_width - gap) // 2
_centered_text(
    draw,
    (0, 0, half_width, top_header),
    "No-MMD training: run 1",
    HEADER_FONT,
)
_centered_text(
    draw,
    (half_width + gap, 0, pair_width, top_header),
    "No-MMD training: run 2 (rerun1)",
    HEADER_FONT,
)
y = top_header
for item in paired.values():
    body = item["image"].crop(
        (0, top_header, item["image"].width, item["image"].height)
    )
    _centered_text(draw, (0, y, pair_width, y + row_header), item["title"], ROW_FONT)
    y += row_header
    combined.paste(body, (0, y))
    y += body.height

COMBINED_PATH = OUTPUT_DIR / "all_results_side_by_side.png"
combined.save(COMBINED_PATH, dpi=(220, 220))

In [ ]:
display(Markdown("## Two no-MMD-loss training results: run 1 (left) vs. run 2 (right)"))
display(Image(filename=str(COMBINED_PATH), width=1200))